# Step 13 — the endotype model, and every sample labelled at its own site

**Data type: RNA_array** (GSE65391). **Reads:** `step12_consensus.rds`, `step11_*`, `step10_site_*`.
**Writes:** `step13_model.rds`, `step13_site_{A,B,C}.rds`.

A central analysis would cut the full consensus matrix into groups. Federated, no one sees the pairs
across sites, so the final partition comes from one more federated k-means at the chosen k, with 50
random starts, on all discovery patients.

**The model** is the gene centre (1,000 numbers) and one **centroid** per endotype: the mean centred
expression of its patients over the 1,000 genes. To build a centroid, each site sends, per cluster,
a count and the sum of its patients' centred gene vectors. Endotypes are named E1, E2, … by size.

**Every sample** is then labelled at its own site by the centroid its centred profile correlates with
best: validation patients, later visits of discovery patients, and healthy children. The **margin** is
the best correlation minus the second best. A small margin means a sample sits between two endotypes.
Each site returns only its counts per endotype.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
source("../src/federation.R")
start_log("13")
f <- readRDS(art("step11_features.rds"))
K <- readRDS(art("step12_consensus.rds"))$k
P <- lapply(setNames(SITES, SITES), function(s) readRDS(site_file("11", s))$scores)
set.seed(SEED + 13L)
fit <- federated_kmeans_best(P, K, rep(0, ncol(P[[1]])), f$sdev, nstart = 50, log = TRUE)
fit$size

[1] 61 49

## Centroids in gene space

In [2]:
cluster_sums <- lapply(SITES, function(s) {
  d <- readRDS(site_file("10", s)); core <- readRDS(site_file("11", s))$core
  Z <- t(d$E[f$genes, core] - f$centre); lab <- fit$labels[[s]]
  send(list(count = tabulate(lab, K),
            sum = t(sapply(seq_len(K), function(j) colSums(Z[lab == j, , drop = FALSE])))),
       s, "per-cluster count and gene sums", length(core))
})
n_k <- Reduce(`+`, lapply(cluster_sums, `[[`, "count"))
centroids <- Reduce(`+`, lapply(cluster_sums, `[[`, "sum")) / n_k
by_size <- order(n_k, decreasing = TRUE)
centroids <- centroids[by_size, , drop = FALSE]
rownames(centroids) <- paste0("E", seq_len(K))
relabel <- setNames(rownames(centroids), by_size)
c(setNames(n_k[by_size], rownames(centroids)))

E1 E2 
61 49

## Each site labels its own samples

In [3]:
agree <- c(); counts <- list()
for (s in SITES) {
  d  <- readRDS(site_file("10", s)); core <- readRDS(site_file("11", s))$core
  nc <- nearest_centroid(t(d$E[f$genes, ] - f$centre), centroids)
  d$meta$endotype  <- nc[rownames(d$meta), "endotype"]
  d$meta$margin    <- nc[rownames(d$meta), "margin"]
  d$meta$clustered <- rownames(d$meta) %in% core
  d$meta$endotype_kmeans <- NA
  d$meta[core, "endotype_kmeans"] <- relabel[as.character(fit$labels[[s]])]
  saveRDS(d, site_file("13", s))
  agree[s] <- send(mean(d$meta[core, "endotype"] == d$meta[core, "endotype_kmeans"]), s,
                   "agreement, k-means vs nearest centroid", ncol(d$E))
  counts[[s]] <- send(table(split = d$meta$split, endotype = d$meta$endotype), s,
                      "counts per split and endotype", ncol(d$E))
}
round(agree, 3)

A     B     C 
1.000 0.973 1.000

In [4]:
Reduce(`+`, counts)
t(sapply(counts, colSums))

            endotype
split         E1  E2
  discovery  340 335
  healthy     48   0
  validation 135 114

,E1,E2
A,179,159
B,155,125
C,189,165


In [5]:
saveRDS(list(genes = f$genes, centre = f$centre, sd = f$sd, centroids = centroids, k = K),
        art("step13_model.rds"))
cat("wrote", art("step13_model.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step13_model.rds 


## Findings

Two endotypes, E1 (61 discovery patients) and E2 (49), the same sizes round 1 found by pooling.
Nearest-centroid labels agree with the k-means partition for 97–100% of discovery patients at every
site. All 48 healthy samples fall in E1. Both endotypes appear at every site in similar proportions:
the site correction of step 06 left no site signal for the clustering to find.